<a href="https://colab.research.google.com/github/wieland-github/information_extraction_german_medical_data/blob/main/gbert_gptnermed_training_base_genutzt_words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline (Off the shelf Model) fine tune auf GPTNERMED
- Hier nutze ich deepset/gbert-base als baseline.
- Model wird auf dem Vorhandenen GPTNERMED fine getuned
- Ergebnisse über drei Seeds gemittelt

In [7]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "evaluate>=0.4.0,<0.5.0"

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import torch
import datasets
import transformers

print(torch.cuda.is_available())

True


## Daten laden

In [10]:
def get_label(label_id):
    label_mapping = {
        0: "Medikation",
        1: "Dosis",
        2: "Diagnose",
    }
    return label_mapping.get(label_id)

print(get_label(0))

Medikation


In [11]:
def get_classes(labels):
    return [get_label(c) for c in labels["ner_class"]]

def to_offset_format(example):
    labels = example["ner_labels"]
    return {
        "text": example["sentence"],
        "ent_starts": labels["start"],
        "ent_stops": labels["stop"],
        "ent_classes": get_classes(labels),
    }

print(to_offset_format({"sentence": "Test", "ner_labels": {"start": [0], "stop": [2], "ner_class": [0]}}))

{'text': 'Test', 'ent_starts': [0], 'ent_stops': [2], 'ent_classes': ['Medikation']}


In [12]:
import datasets

raw = datasets.load_dataset("jfrei/GPTNERMED", trust_remote_code=True)
dataset = raw.map(to_offset_format, remove_columns=["sentence", "ner_labels"])

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} dev and {len(dataset['test'])} test")
print(dataset["train"][0])

Loaded 7876 train, 984 dev and 985 test
{'text': '0,4 Diuretika 0,25 1x/die', 'ent_starts': [0, 4, 14, 19], 'ent_stops': [3, 13, 18, 25], 'ent_classes': ['Dosis', 'Medikation', 'Dosis', 'Dosis']}


## Syntetische Sätze hinzufügen

In [13]:
from datasets import concatenate_datasets

SYNTH_PATH = "/content/drive/MyDrive/synthetic_gptnermed.jsonl"

synthetic = datasets.load_dataset("json", data_files=SYNTH_PATH, split="train")
synthetic = synthetic.map(to_offset_format, remove_columns=["sentence", "ner_labels"])

synthetic = synthetic.cast(dataset["train"].features)

n_before = len(dataset["train"])
dataset["train"] = concatenate_datasets([dataset["train"], synthetic])

print(f"Train: {n_before} -> {len(dataset['train'])} (+{len(synthetic)} synthetisch)")
print(f"Validation: {len(dataset['validation'])}, Test: {len(dataset['test'])} (unveraendert)")
print(dataset["train"][-1])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train: 7876 -> 12876 (+5000 synthetisch)
Validation: 984, Test: 985 (unveraendert)
{'text': 'Der Patient erhält alclofenac in einer Dosierung von 1,5 mg.', 'ent_starts': [19, 53], 'ent_stops': [29, 59], 'ent_classes': ['Medikation', 'Dosis']}


## Daten vorverarbeiten

BIO schema:
- B: Anfang der Entität
- I: Innerhalb der Entität
- O: keine Entität


In [14]:
label_list = ["O", "B-Medikation", "I-Medikation", "B-Dosis", "I-Dosis", "B-Diagnose", "I-Diagnose"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-Medikation', 2: 'I-Medikation', 3: 'B-Dosis', 4: 'I-Dosis', 5: 'B-Diagnose', 6: 'I-Diagnose'}
{'O': 0, 'B-Medikation': 1, 'I-Medikation': 2, 'B-Dosis': 3, 'I-Dosis': 4, 'B-Diagnose': 5, 'I-Diagnose': 6}


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepset/gbert-base")

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [16]:
def get_token_label(start, end, entities):
    if start == end:
        return -100

    for ent_start, ent_stop, ent_class in entities:
        if ent_start <= start and end <= ent_stop:
            prefix = "B-" if start == ent_start else "I-"
            return label2id[prefix + ent_class]

    return label2id["O"]

print(get_token_label(0, 4, [(0, 8, "Medikation")]))
print(get_token_label(3, 8, [(0, 8, "Medikation")]))

1
2


In [17]:
def get_entities(example):
    return list(zip(example["ent_starts"], example["ent_stops"], example["ent_classes"]))

print(get_entities(dataset["train"][0]))

[(0, 3, 'Dosis'), (4, 13, 'Medikation'), (14, 18, 'Dosis'), (19, 25, 'Dosis')]


In [18]:
def align_labels(offsets, entities):
    return [get_token_label(start, end, entities) for start, end in offsets]

enc = tokenizer("Aspirin 500mg", return_offsets_mapping=True)
print(align_labels(enc["offset_mapping"], [(0, 7, "Medikation"), (8, 13, "Dosis")]))


[-100, 1, 2, 2, 3, 4, -100]


In [19]:
def preprocess_function(example):
    tokenized = tokenizer(example["text"], truncation=True, max_length=128, return_offsets_mapping=True)
    entities = get_entities(example)
    tokenized["labels"] = align_labels(tokenized.pop("offset_mapping"), entities)
    return tokenized

print(preprocess_function(dataset["train"][0]))


{'input_ids': [102, 430, 818, 470, 2182, 1947, 23816, 430, 818, 1693, 164, 30950, 1061, 128, 103], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, 4, 4, 1, 2, 2, 3, 4, 4, 3, 4, 4, 4, -100]}


In [20]:
tokenized_dataset = dataset.map(
    preprocess_function,
    remove_columns=["text", "ent_starts", "ent_stops", "ent_classes"],
)
tokenized_dataset = tokenized_dataset.shuffle(seed=42)


Map:   0%|          | 0/12876 [00:00<?, ? examples/s]

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

In [21]:
example = tokenized_dataset["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
for token, label in zip(tokens, example["labels"]):
    print(token, "->", id2label[label] if label != -100 else "-100")

[CLS] -> -100
Bei -> O
der -> O
Hinweis -> O
##geber -> O
##in -> O
wurde -> O
die -> O
Diagnose -> O
einer -> O
P -> B-Diagnose
##neum -> I-Diagnose
##onie -> I-Diagnose
gestellt -> O
. -> O
[SEP] -> -100


In [22]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [23]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred, lab in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                sentence_preds.append(label_list[p])
                sentence_labels.append(label_list[l])
        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

    for entity in ["Medikation", "Dosis", "Diagnose"]:
      if entity in results:
          metrics[f"{entity}_precision"] = results[entity]["precision"]
          metrics[f"{entity}_recall"] = results[entity]["recall"]
          metrics[f"{entity}_f1"] = results[entity]["f1"]

    return metrics

## Training (über drei Seeds)

In [24]:
import wandb
wandb.init(mode="disabled")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [25]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, set_seed

seeds = [42, 72, 102]
seed_results = []

for seed in seeds:
    print(f"\n===== Training mit Seed {seed} =====")
    set_seed(seed)

    model = AutoModelForTokenClassification.from_pretrained(
        "deepset/gbert-base", num_labels=7, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"gbert_gptnermed_seed{seed}",
        evaluation_strategy="epoch",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_metrics = trainer.predict(tokenized_dataset["test"]).metrics
    test_metrics["seed"] = seed
    seed_results.append(test_metrics)
    print(f"Seed {seed}: Test-F1 = {test_metrics['test_f1']:.4f}")


===== Training mit Seed 42 =====


model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.133300,0.205949,0.835076,0.880781,0.857320,0.931925,0.896552,0.905738,0.901121,0.844761,0.876510,0.860343,0.741403,0.847484,0.790902
2,0.082200,0.205325,0.864773,0.865507,0.865140,0.938508,0.924708,0.893443,0.908807,0.893939,0.871141,0.882393,0.752174,0.816038,0.782805
3,0.040100,0.206423,0.881738,0.895206,0.888421,0.946009,0.929592,0.933402,0.931493,0.898236,0.888591,0.893387,0.794379,0.844340,0.818598


Seed 42: Test-F1 = 0.8982

===== Training mit Seed 72 =====


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.132600,0.188716,0.849590,0.879508,0.864290,0.937436,0.886497,0.928279,0.906907,0.890983,0.888591,0.889785,0.748148,0.794025,0.770404
2,0.082200,0.213871,0.884064,0.889690,0.886868,0.941876,0.937173,0.917008,0.926981,0.916549,0.869799,0.892562,0.780282,0.871069,0.823180
3,0.039300,0.229499,0.884937,0.897327,0.891089,0.944478,0.921827,0.930328,0.926058,0.923611,0.892617,0.907850,0.791241,0.852201,0.820590


Seed 72: Test-F1 = 0.9026

===== Training mit Seed 102 =====


Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Medikation Precision,Medikation Recall,Medikation F1,Dosis Precision,Dosis Recall,Dosis F1,Diagnose Precision,Diagnose Recall,Diagnose F1
1,0.139700,0.205437,0.858165,0.865083,0.861610,0.934476,0.925000,0.909836,0.917355,0.895105,0.859060,0.876712,0.728959,0.803459,0.764398
2,0.080100,0.210757,0.855114,0.893933,0.874093,0.937998,0.907000,0.929303,0.918016,0.882038,0.883221,0.882629,0.754875,0.852201,0.800591
3,0.042800,0.214096,0.873439,0.890115,0.881698,0.945652,0.921668,0.928279,0.924962,0.900680,0.888591,0.894595,0.774854,0.833333,0.803030


Seed 102: Test-F1 = 0.8985


## Ergebnisse

In [26]:
import numpy as np

print("Test-Ergebnisse pro Seed:")
for r in seed_results:
    print(f"  Seed {r['seed']}: F1={r['test_f1']:.4f}  Precision={r['test_precision']:.4f}  Recall={r['test_recall']:.4f}")

f1_scores = [r["test_f1"] for r in seed_results]
prec_scores = [r["test_precision"] for r in seed_results]
rec_scores = [r["test_recall"] for r in seed_results]

print("\nÜber alle Seeds gemittelt (Mittelwert ± Std):")
print(f"  F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"  Recall:    {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")

Test-Ergebnisse pro Seed:
  Seed 42: F1=0.8982  Precision=0.8939  Recall=0.9026
  Seed 72: F1=0.9026  Precision=0.8981  Recall=0.9072
  Seed 102: F1=0.8985  Precision=0.8903  Recall=0.9068

Über alle Seeds gemittelt (Mittelwert ± Std):
  F1:        0.8998 ± 0.0020
  Precision: 0.8941 ± 0.0032
  Recall:    0.9055 ± 0.0021


In [27]:
categories = ["Diagnose", "Medikation", "Dosis"]

metrics = {
    "F1": "f1",
    "Precision": "precision",
    "Recall": "recall",
}

print("Verschiedene Kategorien über alle Seeds gemittelt (Mittelwert ± Std):")

for category in categories:
    print(f"\n{category}:")

    for metric_name, metric_key in metrics.items():
        scores = [
            r[f"test_{category}_{metric_key}"]
            for r in seed_results
        ]

        print(
            f"  {metric_name:10s}: "
            f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
        )

Verschiedene Kategorien über alle Seeds gemittelt (Mittelwert ± Std):

Diagnose:
  F1        : 0.8294 ± 0.0012
  Precision : 0.8207 ± 0.0056
  Recall    : 0.8383 ± 0.0034

Medikation:
  F1        : 0.9380 ± 0.0014
  Precision : 0.9361 ± 0.0020
  Recall    : 0.9398 ± 0.0009

Dosis:
  F1        : 0.9057 ± 0.0070
  Precision : 0.8980 ± 0.0077
  Recall    : 0.9136 ± 0.0070
